# 面试问题：Greedy、Beam Search、Top-k 和 Top-p 解码如何实现与选择？

可以直接复述的回答是：Greedy 每步取最大概率 token，延迟低但可能被局部最优困住。Beam Search 同时保留若干高分前缀，适合翻译或结构化生成等“找高概率序列”任务，但宽度增大会增加计算并可能偏好短句。Top-k 只在概率最高的固定 k 个 token 中采样，尾部大小随上下文不变。Top-p 则选择累计概率达到 p 的最小集合，候选数会随分布尖锐程度自适应变化。采样结果必须固定随机种子才能复现，安全或格式约束还应在候选过滤层实现。下面从条件概率表手写四种解码。

## 真实案例：五个业务提示词的短回复生成

五个提示覆盖退款答复、物流状态、告警摘要、SQL 补全和标题生成。条件概率为教学构造的小语言模型输出，每个 token 都是可读词片；它用于解释搜索树，不代表真实大模型质量。

In [1]:
import math  # 导入对数概率计算函数
import random  # 导入可复现随机采样器
cases = [  # 定义五个带期望序列的业务提示
    {"id": "D-01", "prompt": "回复：订单退款已完成", "good": ["订单", "已退款", "<eos>"]},  # 构造 Greedy 局部最优反例
    {"id": "D-02", "prompt": "物流状态：包裹到站", "good": ["包裹", "待取件", "<eos>"]},  # 物流通知生成
    {"id": "D-03", "prompt": "告警摘要：磁盘 95%", "good": ["磁盘", "空间不足", "<eos>"]},  # 运维摘要生成
    {"id": "D-04", "prompt": "SQL 补全：SELECT", "good": ["列名", "FROM", "<eos>"]},  # 代码补全生成
    {"id": "D-05", "prompt": "标题：夏季咖啡活动", "good": ["夏日", "咖啡节", "<eos>"]},  # 营销标题生成
]  # 结束五个生成案例
tables = {  # 定义每个案例的前缀到下一 token 概率表
    "D-01": {(): {"抱歉": 0.55, "订单": 0.45}, ("抱歉",): {"您的": 0.51, "<eos>": 0.49}, ("抱歉", "您的"): {"问题": 0.51, "<eos>": 0.49}, ("抱歉", "您的", "问题"): {"<eos>": 1.0}, ("订单",): {"已退款": 0.90, "<eos>": 0.10}, ("订单", "已退款"): {"<eos>": 1.0}},  # 局部高概率开头对应低联合概率路径
    "D-02": {(): {"包裹": 0.72, "快递": 0.28}, ("包裹",): {"待取件": 0.88, "运输中": 0.12}, ("快递",): {"运输中": 0.60, "<eos>": 0.40}, ("包裹", "待取件"): {"<eos>": 1.0}, ("包裹", "运输中"): {"<eos>": 1.0}, ("快递", "运输中"): {"<eos>": 1.0}},  # 尖锐物流分布
    "D-03": {(): {"磁盘": 0.68, "服务器": 0.32}, ("磁盘",): {"空间不足": 0.82, "需关注": 0.18}, ("服务器",): {"需关注": 0.70, "<eos>": 0.30}, ("磁盘", "空间不足"): {"<eos>": 1.0}, ("磁盘", "需关注"): {"<eos>": 1.0}, ("服务器", "需关注"): {"<eos>": 1.0}},  # 告警摘要分布
    "D-04": {(): {"列名": 0.65, "星号": 0.35}, ("列名",): {"FROM": 0.90, "WHERE": 0.10}, ("星号",): {"FROM": 0.85, "<eos>": 0.15}, ("列名", "FROM"): {"<eos>": 1.0}, ("列名", "WHERE"): {"<eos>": 1.0}, ("星号", "FROM"): {"<eos>": 1.0}},  # 结构化补全分布
    "D-05": {(): {"夏日": 0.52, "清凉": 0.30, "限定": 0.18}, ("夏日",): {"咖啡节": 0.70, "特饮": 0.30}, ("清凉",): {"特饮": 0.65, "咖啡节": 0.35}, ("限定",): {"特饮": 0.80, "<eos>": 0.20}, ("夏日", "咖啡节"): {"<eos>": 1.0}, ("夏日", "特饮"): {"<eos>": 1.0}, ("清凉", "特饮"): {"<eos>": 1.0}, ("清凉", "咖啡节"): {"<eos>": 1.0}, ("限定", "特饮"): {"<eos>": 1.0}},  # 较平坦的创意标题分布
}  # 结束五个条件概率树
print("输入预览：id | prompt | 期望高联合概率序列")  # 输出生成任务标题
for case in cases:  # 逐条展示五个业务提示
    print(f"{case['id']} | {case['prompt']} | {' '.join(case['good'])}")  # 展示提示和评估目标
print("D-01 首 token 分布：", tables["D-01"][()])  # 展示局部最优反例的第一步概率

输入预览：id | prompt | 期望高联合概率序列
D-01 | 回复：订单退款已完成 | 订单 已退款 <eos>
D-02 | 物流状态：包裹到站 | 包裹 待取件 <eos>
D-03 | 告警摘要：磁盘 95% | 磁盘 空间不足 <eos>
D-04 | SQL 补全：SELECT | 列名 FROM <eos>
D-05 | 标题：夏季咖啡活动 | 夏日 咖啡节 <eos>
D-01 首 token 分布： {'抱歉': 0.55, '订单': 0.45}


## Baseline / 基线：Greedy 每步只取最大概率 token

Greedy 不回看已放弃分支。D-01 首步选择 0.55 的“抱歉”，但后续每步只有 0.51；以 0.45 开头的“订单→已退款”整体概率反而更高。

In [2]:
def next_distribution(case_id, prefix):  # 查询给定案例和前缀的下一 token 分布
    return tables[case_id].get(tuple(prefix), {"<eos>": 1.0})  # 未定义前缀安全结束生成
def greedy_decode(case_id, max_steps=5):  # 手写逐步最大概率解码
    sequence = []  # 初始化空生成序列
    log_probability = 0.0  # 累计整条序列的对数概率
    trace = []  # 保存每步候选和选择
    for step in range(max_steps):  # 限制最大步数防止无限生成
        distribution = next_distribution(case_id, sequence)  # 获取当前前缀条件分布
        token = max(distribution, key=distribution.get)  # 选择单步概率最大 token
        probability = distribution[token]  # 读取所选 token 概率
        log_probability += math.log(probability)  # 累加联合对数概率
        sequence.append(token)  # 把 token 追加到前缀
        trace.append((step + 1, distribution, token, log_probability))  # 保存当前搜索轨迹
        if token == "<eos>":  # 检查是否生成结束标记
            break  # 遇到结束标记立即停止
    return sequence, log_probability, trace  # 返回序列、联合分数和逐步轨迹
greedy_results = {case["id"]: greedy_decode(case["id"]) for case in cases}  # 对五个提示执行 Greedy 基线
print("D-01 Greedy 轨迹：step | distribution | choose | cumulative logp")  # 输出局部最优路径表头
for step, distribution, token, score in greedy_results["D-01"][2]:  # 遍历 D-01 每步决策
    print(f"{step} | {distribution} | {token} | {score:.4f}")  # 展示只保留一个前缀的后果
print("Greedy 结果：", greedy_results["D-01"][0])  # 展示错误的局部最优完整序列

D-01 Greedy 轨迹：step | distribution | choose | cumulative logp
1 | {'抱歉': 0.55, '订单': 0.45} | 抱歉 | -0.5978
2 | {'您的': 0.51, '<eos>': 0.49} | 您的 | -1.2712
3 | {'问题': 0.51, '<eos>': 0.49} | 问题 | -1.9445
4 | {'<eos>': 1.0} | <eos> | -1.9445
Greedy 结果： ['抱歉', '您的', '问题', '<eos>']


## 核心实现：Beam Search 保留多个前缀

Beam 宽度为 2，每步展开所有保留前缀，再按累计 log probability 截断。教学例所有候选长度相近，暂不加 length penalty。

In [3]:
def beam_search(case_id, beam_width=2, max_steps=5):  # 手写固定宽度束搜索
    beams = [([], 0.0)]  # 用空前缀和零对数概率初始化搜索束
    frontier_trace = []  # 保存每轮截断后的候选前缀
    for step in range(max_steps):  # 最多展开指定 token 步数
        candidates = []  # 收集本轮所有束的扩展结果
        for sequence, score in beams:  # 遍历当前保留的所有前缀
            if sequence and sequence[-1] == "<eos>":  # 已结束序列不再继续扩展
                candidates.append((sequence, score))  # 原样保留已完成候选
                continue  # 转向下一条束
            distribution = next_distribution(case_id, sequence)  # 获取当前前缀下一 token 概率
            for token, probability in distribution.items():  # 枚举该前缀所有合法 token
                candidates.append((sequence + [token], score + math.log(probability)))  # 形成新前缀和联合对数概率
        candidates.sort(key=lambda item: item[1], reverse=True)  # 按联合分数从高到低排序
        beams = candidates[:beam_width]  # 只保留指定数量的最优前缀
        frontier_trace.append([(sequence.copy(), score) for sequence, score in beams])  # 保存本轮搜索前沿
        if all(sequence[-1] == "<eos>" for sequence, score in beams):  # 检查所有束是否已经结束
            break  # 全部完成时提前终止搜索
    return beams[0][0], beams[0][1], frontier_trace  # 返回最高分完整序列和搜索轨迹
beam_results = {case["id"]: beam_search(case["id"]) for case in cases}  # 对五个提示执行 Beam Search
print("D-01 Beam 前沿：")  # 输出束搜索中间过程标题
for step, frontier in enumerate(beam_results["D-01"][2], start=1):  # 逐轮遍历保留前缀
    print(f"step={step} -> {[(tokens, round(score, 4)) for tokens, score in frontier]}")  # 展示多个分支竞争过程
print("Beam 结果：", beam_results["D-01"][0], "logp=", round(beam_results["D-01"][1], 4))  # 展示跳出局部最优后的序列

D-01 Beam 前沿：
step=1 -> [(['抱歉'], -0.5978), (['订单'], -0.7985)]
step=2 -> [(['订单', '已退款'], -0.9039), (['抱歉', '您的'], -1.2712)]
step=3 -> [(['订单', '已退款', '<eos>'], -0.9039), (['抱歉', '您的', '问题'], -1.9445)]
step=4 -> [(['订单', '已退款', '<eos>'], -0.9039), (['抱歉', '您的', '问题', '<eos>'], -1.9445)]
Beam 结果： ['订单', '已退款', '<eos>'] logp= -0.9039


## Top-k 与 Top-p：在过滤后的分布中采样

Top-k 固定候选数量；Top-p 按概率降序累加直到达到阈值。使用独立随机种子展示可复现多样性。

In [4]:
def filter_distribution(distribution, mode, value):  # 手写 Top-k 或 Top-p 候选过滤
    ordered = sorted(distribution.items(), key=lambda item: item[1], reverse=True)  # 按概率从高到低排序 token
    if mode == "topk":  # 处理固定候选数过滤
        kept = ordered[:int(value)]  # 截取概率最高的 k 个 token
    else:  # 处理累计概率阈值过滤
        kept = []  # 初始化 nucleus 候选集合
        cumulative = 0.0  # 初始化累计原始概率
        for token, probability in ordered:  # 从最大概率 token 开始累加
            kept.append((token, probability))  # 将当前 token 纳入候选集合
            cumulative += probability  # 更新累计概率质量
            if cumulative >= float(value):  # 达到 p 阈值时停止扩展集合
                break  # 使用满足阈值的最小候选集
    total = sum(probability for token, probability in kept)  # 计算过滤后归一化常数
    return [(token, probability / total) for token, probability in kept]  # 返回重新归一化的候选分布
def sample_decode(case_id, mode, value, seed, max_steps=5):  # 在过滤分布中执行可复现采样
    generator = random.Random(seed)  # 创建不污染全局状态的随机数生成器
    sequence = []  # 初始化采样序列
    for step in range(max_steps):  # 限制生成最大长度
        filtered = filter_distribution(next_distribution(case_id, sequence), mode, value)  # 计算当前过滤候选集合
        draw = generator.random()  # 生成零到一之间的固定随机数
        cumulative = 0.0  # 初始化归一化累计概率
        token = filtered[-1][0]  # 预设浮点边界下的最后候选
        for candidate, probability in filtered:  # 按候选概率区间定位抽样 token
            cumulative += probability  # 更新累计概率上界
            if draw <= cumulative:  # 检查随机数是否落入当前区间
                token = candidate  # 选择命中的 token
                break  # 结束当前 token 抽样
        sequence.append(token)  # 把采样 token 加入前缀
        if token == "<eos>":  # 检查是否采到结束标记
            break  # 完成当前序列生成
    return sequence  # 返回完整采样序列
topk_samples = [sample_decode("D-05", "topk", 2, seed) for seed in range(5)]  # 用五个种子采样 Top-k 标题
topp_samples = [sample_decode("D-05", "topp", 0.8, seed) for seed in range(5)]  # 用五个种子采样 nucleus 标题
print("D-05 首步 Top-k(k=2)：", filter_distribution(tables["D-05"][()], "topk", 2))  # 展示固定两个候选的归一化分布
print("D-05 首步 Top-p(p=0.8)：", filter_distribution(tables["D-05"][()], "topp", 0.8))  # 展示累计质量决定的候选集合
print("Top-k 五次可复现样本：", topk_samples)  # 展示固定候选采样多样性
print("Top-p 五次可复现样本：", topp_samples)  # 展示自适应候选采样多样性

D-05 首步 Top-k(k=2)： [('夏日', 0.6341463414634146), ('清凉', 0.3658536585365853)]
D-05 首步 Top-p(p=0.8)： [('夏日', 0.6341463414634146), ('清凉', 0.3658536585365853)]
Top-k 五次可复现样本： [['清凉', '咖啡节', '<eos>'], ['夏日', '特饮', '<eos>'], ['清凉', '咖啡节', '<eos>'], ['夏日', '咖啡节', '<eos>'], ['夏日', '咖啡节', '<eos>']]
Top-p 五次可复现样本： [['清凉', '咖啡节', '<eos>'], ['夏日', '特饮', '<eos>'], ['清凉', '咖啡节', '<eos>'], ['夏日', '咖啡节', '<eos>'], ['夏日', '咖啡节', '<eos>']]


## 失败案例与修正、逐样本结果

D-01 是可复现失败：Greedy 的首 token 概率更高，但完整序列概率更低。Beam 保留第二个前缀后找回正确高联合概率序列。其余案例用于确认没有为了一个反例破坏常规生成。

In [5]:
greedy_exact = 0  # 初始化 Greedy 序列命中数
beam_exact = 0  # 初始化 Beam 序列命中数
print("id | Greedy | Greedy logp | Beam | Beam logp | 期望")  # 输出五案例对照表头
for case in cases:  # 遍历五个业务提示词
    greedy_sequence, greedy_score, greedy_trace = greedy_results[case["id"]]  # 读取 Greedy 完整结果
    beam_sequence, beam_score, beam_trace = beam_results[case["id"]]  # 读取 Beam 完整结果
    greedy_exact += int(greedy_sequence == case["good"])  # 累加 Greedy 精确命中
    beam_exact += int(beam_sequence == case["good"])  # 累加 Beam 精确命中
    print(f"{case['id']} | {' '.join(greedy_sequence):14} | {greedy_score:10.4f} | {' '.join(beam_sequence):14} | {beam_score:9.4f} | {' '.join(case['good'])}")  # 展示逐提示搜索结果
greedy_accuracy = greedy_exact / len(cases)  # 计算 Greedy 精确序列准确率
beam_accuracy = beam_exact / len(cases)  # 计算 Beam 精确序列准确率
failure_gain = math.exp(beam_results["D-01"][1]) / math.exp(greedy_results["D-01"][1])  # 计算失败样本联合概率提升倍数
print(f"Greedy exact={greedy_accuracy:.1%}，Beam exact={beam_accuracy:.1%}，D-01 联合概率提升={failure_gain:.2f}x")  # 汇总同数据对照与修正收益

id | Greedy | Greedy logp | Beam | Beam logp | 期望
D-01 | 抱歉 您的 问题 <eos> |    -1.9445 | 订单 已退款 <eos>   |   -0.9039 | 订单 已退款 <eos>
D-02 | 包裹 待取件 <eos>   |    -0.4563 | 包裹 待取件 <eos>   |   -0.4563 | 包裹 待取件 <eos>
D-03 | 磁盘 空间不足 <eos>  |    -0.5841 | 磁盘 空间不足 <eos>  |   -0.5841 | 磁盘 空间不足 <eos>
D-04 | 列名 FROM <eos>  |    -0.5361 | 列名 FROM <eos>  |   -0.5361 | 列名 FROM <eos>
D-05 | 夏日 咖啡节 <eos>   |    -1.0106 | 夏日 咖啡节 <eos>   |   -1.0106 | 夏日 咖啡节 <eos>
Greedy exact=80.0%，Beam exact=100.0%，D-01 联合概率提升=2.83x


## 结果解读

Beam 在 D-01 第一轮没有丢掉“订单”分支，第二轮后其累计分数超过“抱歉”路径，因此修复了局部最优。Top-k 与 Top-p 的输出不是“更准确”的替代物，而是面向开放式任务的多样性策略；分布越尖锐，Top-p 保留的 token 越少。

## 生产边界

真实模型词表数万、KV Cache 显存有限，还需 length penalty、重复惩罚、停止词、批量 beam 重排和约束解码。采样质量应以人工偏好、安全率与任务成功率评估，不能用本例五条 exact match 外推。线上复现还需保存模型版本、tokenizer、温度、随机种子和完整解码参数。

## 最小回归测试

In [6]:
assert len(cases) >= 5  # 保证生成实验覆盖至少五个业务提示
assert greedy_results["D-01"][0] != cases[0]["good"]  # 保证 Greedy 局部最优失败真实复现
assert beam_results["D-01"][0] == cases[0]["good"]  # 保证 Beam 保留分支后修复失败
assert beam_results["D-01"][1] > greedy_results["D-01"][1]  # 保证修正序列联合概率确实更高
assert beam_accuracy > greedy_accuracy  # 保证同一五案例下 Beam 精确命中更高
assert len(filter_distribution(tables["D-05"][()], "topk", 2)) == 2  # 保证 Top-k 固定候选数量
assert 0.8 <= sum(tables["D-05"][()][token] for token, probability in filter_distribution(tables["D-05"][()], "topp", 0.8)) <= 1.0  # 保证 Top-p 原始累计概率达到阈值